#lab 06

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import kagglehub

# Download latest ver"sion
path = kagglehub.dataset_download("shreyapmaher/fruits-dataset-images")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'fruits-dataset-images' dataset.
Path to dataset files: /kaggle/input/fruits-dataset-images


#new try


In [ ]:
# ============================================================
# INSTALL REQUIRED LIBRARIES
# ============================================================

!pip install split-folders -q
!pip install openpyxl -q

# ============================================================
# IMPORTS
# ============================================================

import os
import pickle
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import splitfolders

import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================================================
# 🍎 COMPLETE IMAGE CLASSIFICATION RESEARCH PIPELINE
# ============================================================

# ============================================================
# 1. IMPORTS
# ============================================================


import os
import pickle
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import splitfolders

import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf

from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    Flatten,
    Conv2D,
    MaxPooling2D,
    GlobalAveragePooling2D
)

from tensorflow.keras.preprocessing.image import ImageDataGenerator

from tensorflow.keras.applications import (
    ResNet50,
    VGG16,
    MobileNetV2
)

from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_pre
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_pre
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobile_pre

from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    auc
)

from sklearn.preprocessing import label_binarize
# ============================================================
# 3. DOWNLOAD DATASET
# ============================================================

import kagglehub

path = kagglehub.dataset_download(
    "shreyapmaher/fruits-dataset-images"
)

print("✅ Dataset Path:", path)

# ============================================================
# 4. ORIGINAL DATASET PATH
# ============================================================

dataset_path = "/kaggle/input/fruits-dataset-images/images"

# ============================================================
# 5. TRAIN-TEST SPLIT
# ============================================================

output_path = "/kaggle/working/fruits_split"

splitfolders.ratio(
    dataset_path,
    output=output_path,
    seed=42,
    ratio=(0.8, 0.2)
)

print("✅ Dataset Split Completed")

# ============================================================
# 2. DATASET PATHS
# ============================================================

train_path = "/kaggle/working/fruits_split/train"
test_path = "/kaggle/working/fruits_split/val"

# ============================================================
# 3. SAVE DIRECTORIES
# ============================================================

BASE_DIR = "/content/drive/MyDrive/Fruits"

MODEL_DIR = os.path.join(BASE_DIR, "models")
RESULT_DIR = os.path.join(BASE_DIR, "results")
HISTORY_DIR = os.path.join(BASE_DIR, "histories")

PLOT_DIR = os.path.join(BASE_DIR, "plots")

CONFUSION_DIR = os.path.join(PLOT_DIR, "confusion")
ROC_DIR = os.path.join(PLOT_DIR, "roc")
CURVE_DIR = os.path.join(PLOT_DIR, "curves")
COMPARE_DIR = os.path.join(PLOT_DIR, "comparisons")

for path in [
    MODEL_DIR,
    RESULT_DIR,
    HISTORY_DIR,
    CONFUSION_DIR,
    ROC_DIR,
    CURVE_DIR,
    COMPARE_DIR
]:
    os.makedirs(path, exist_ok=True)

# ============================================================
# 4. CONFIG
# ============================================================

IMG_SIZE = (224,224)

epochs_list = [10]

learning_rates = [0.0001]

batch_sizes = [24]

optimizers = {
    "Adam": tf.keras.optimizers.Adam,
    "RMSprop": tf.keras.optimizers.RMSprop
}

transfer_models = {
    "ResNet50": (ResNet50, resnet_pre),
    "MobileNetV2": (MobileNetV2, mobile_pre)
}

# ============================================================
# 5. GENERATORS
# ============================================================

def get_generators(preprocess_func,
                   batch_size,
                   augmentation=False):

    if augmentation:

        train_datagen = ImageDataGenerator(
            preprocessing_function=preprocess_func,
            rotation_range=20,
            zoom_range=0.2,
            horizontal_flip=True
        )

    else:

        train_datagen = ImageDataGenerator(
            preprocessing_function=preprocess_func
        )

    test_datagen = ImageDataGenerator(
        preprocessing_function=preprocess_func
    )

    train_gen = train_datagen.flow_from_directory(
        train_path,
        target_size=IMG_SIZE,
        batch_size=batch_size,
        class_mode='categorical'
    )

    test_gen = test_datagen.flow_from_directory(
        test_path,
        target_size=IMG_SIZE,
        batch_size=batch_size,
        class_mode='categorical',
        shuffle=False
    )

    return train_gen, test_gen

# ============================================================
# 6. DNN MODEL
# ============================================================

def build_dnn(num_classes, optimizer_class, lr):

    model = Sequential([

        Flatten(input_shape=(224,224,3)),

        Dense(512, activation='relu'),
        Dropout(0.5),

        Dense(256, activation='relu'),
        Dropout(0.5),

        Dense(num_classes, activation='softmax')
    ])

    optimizer = optimizer_class(learning_rate=lr)

    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

# ============================================================
# 7. CNN MODEL
# ============================================================

def build_cnn(num_classes, optimizer_class, lr):

    model = Sequential([

        Conv2D(32, (3,3),
                 activation='relu',
                 input_shape=(224,224,3)),

        MaxPooling2D(2,2),

        Conv2D(64, (3,3),
                 activation='relu'),

        MaxPooling2D(2,2),

        Conv2D(128, (3,3),
                 activation='relu'),

        MaxPooling2D(2,2),

        Flatten(),

        Dense(256, activation='relu'),

        Dropout(0.5),

        Dense(num_classes,
              activation='softmax')
    ])

    optimizer = optimizer_class(learning_rate=lr)

    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

# ============================================================
# 8. TRANSFER LEARNING MODEL
# ============================================================

def build_transfer_model(base_model_fn,
                         num_classes,
                         optimizer_class,
                         lr,
                         freeze=True):

    base = base_model_fn(
        weights='imagenet',
        include_top=False,
        input_shape=(224,224,3)
    )

    if freeze:

        for layer in base.layers:
            layer.trainable = False

    else:

        for layer in base.layers[:-20]:
            layer.trainable = False

        for layer in base.layers[-20:]:
            layer.trainable = True

    x = GlobalAveragePooling2D()(base.output)

    x = Dense(128, activation='relu')(x)

    x = Dropout(0.5)(x)

    output = Dense(num_classes,
                   activation='softmax')(x)

    model = Model(base.input, output)

    optimizer = optimizer_class(
        learning_rate=lr
    )

    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

# ============================================================
# 9. METRICS
# ============================================================

def compute_metrics(y_true, y_pred, y_proba):

    precision = precision_score(
        y_true,
        y_pred,
        average='macro'
    )

    recall = recall_score(
        y_true,
        y_pred,
        average='macro'
    )

    f1 = f1_score(
        y_true,
        y_pred,
        average='macro'
    )

    roc_auc = roc_auc_score(
        label_binarize(y_true,
        classes=np.unique(y_true)),
        y_proba,
        multi_class='ovr'
    )

    cm = confusion_matrix(y_true, y_pred)

    TP = np.diag(cm)

    FP = cm.sum(axis=0) - TP

    FN = cm.sum(axis=1) - TP

    TN = cm.sum() - (FP + FN + TP)

    specificity = np.mean(TN / (TN + FP))

    fpr = np.mean(FP / (FP + TN))

    fnr = np.mean(FN / (FN + TP))

    error_rate = np.mean(FP + FN) / np.sum(cm)

    return precision, recall, f1, specificity, fpr, fnr, error_rate, roc_auc

# ============================================================
# 10. PLOTTING
# ============================================================

def plot_all(y_true,
             y_pred,
             y_proba,
             history,
             class_names,
             title):

    # =========================
    # CONFUSION MATRIX
    # =========================

    plt.figure(figsize=(8,6))

    cm = confusion_matrix(y_true, y_pred)

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names
    )

    plt.title("Confusion Matrix")

    plt.savefig(
        os.path.join(CONFUSION_DIR,
        f"{title}.png")
    )

    plt.close()

    # =========================
    # ROC
    # =========================

    plt.figure(figsize=(8,6))

    y_bin = label_binarize(
        y_true,
        classes=range(len(class_names))
    )

    for i in range(len(class_names)):

        fpr, tpr, _ = roc_curve(
            y_bin[:, i],
            y_proba[:, i]
        )

        roc_auc = auc(fpr, tpr)

        if i < 5:

            plt.plot(
                fpr,
                tpr,
                label=f"{class_names[i]} ({roc_auc:.2f})"
            )

    plt.plot([0,1],[0,1],'k--')

    plt.legend()

    plt.title("ROC Curve")

    plt.savefig(
        os.path.join(ROC_DIR,
        f"{title}.png")
    )

    plt.close()

    # =========================
    # CURVES
    # =========================

    fig, ax = plt.subplots(1,2,
                           figsize=(12,5))

    ax[0].plot(history.history['accuracy'])
    ax[0].plot(history.history['val_accuracy'])

    ax[0].set_title("Accuracy")

    ax[1].plot(history.history['loss'])
    ax[1].plot(history.history['val_loss'])

    ax[1].set_title("Loss")

    plt.savefig(
        os.path.join(CURVE_DIR,
        f"{title}.png")
    )

    plt.close()

# ============================================================
# 11. TRAIN FUNCTION
# ============================================================

results = []

def train_model(model_name,
                model_builder,
                preprocess_func=None,
                transfer=False):

    global results

    for augmentation in [False, True]:

        aug_name = (
            "Augmented"
            if augmentation
            else "NoAug"
        )

        freeze_modes = [True, False] if transfer else [False]

        for freeze in freeze_modes:

            freeze_name = (
                "Frozen"
                if freeze
                else "FineTune"
            )

            for opt_name, opt_class in optimizers.items():

                for lr in learning_rates:

                    for bs in batch_sizes:

                        for epochs in epochs_list:

                            file_name = (
                                f"{model_name}_"
                                f"{aug_name}_"
                                f"{freeze_name}_"
                                f"{opt_name}_"
                                f"lr{lr}_"
                                f"bs{bs}_"
                                f"ep{epochs}.keras"
                            )

                            model_path = os.path.join(
                                MODEL_DIR,
                                file_name
                            )

                            # =====================
                            # SKIP IF EXISTS
                            # =====================

                           # ============================================
                            # SMART SKIP SYSTEM
                            # ============================================

                            existing_models = os.listdir(MODEL_DIR)

                            skip_model = False

                            for existing in existing_models:

                                # Skip if same core configuration exists
                                if (
                                    model_name in existing and
                                    aug_name in existing and
                                    freeze_name in existing and
                                    opt_name in existing and
                                    f"bs{bs}" in existing
                                ):

                                    skip_model = True
                                    print(f"⏩ Already Trained -> {existing}")
                                    break

                            if skip_model:
                                continue
                            # =====================
                            # GENERATORS
                            # =====================

                            preprocess = (
                                preprocess_func
                                if preprocess_func
                                else lambda x: x/255.0
                            )

                            train_gen, test_gen = get_generators(
                                preprocess,
                                bs,
                                augmentation
                            )

                            class_names = list(
                                train_gen.class_indices.keys()
                            )

                            # =====================
                            # MODEL
                            # =====================

                            if transfer:

                                model = model_builder(
                                    train_gen.num_classes,
                                    opt_class,
                                    lr,
                                    freeze
                                )

                            else:

                                model = model_builder(
                                    train_gen.num_classes,
                                    opt_class,
                                    lr
                                )

                            # =====================
                            # TRAIN
                            # =====================

                            history = model.fit(
                                train_gen,
                                validation_data=test_gen,
                                epochs=epochs,
                                verbose=1
                            )

                            # =====================
                            # SAVE MODEL
                            # =====================

                            model.save(model_path)

                            # =====================
                            # SAVE HISTORY
                            # =====================

                            with open(
                                os.path.join(
                                    HISTORY_DIR,
                                    file_name.replace(
                                        ".keras",
                                        ".pkl"
                                    )
                                ),
                                "wb"
                            ) as f:

                                pickle.dump(
                                    history.history,
                                    f
                                )

                            # =====================
                            # PREDICTIONS
                            # =====================

                            y_proba = model.predict(test_gen)

                            y_pred = np.argmax(
                                y_proba,
                                axis=1
                            )

                            y_true = test_gen.classes

                            # =====================
                            # METRICS
                            # =====================

                            loss, acc = model.evaluate(
                                test_gen,
                                verbose=0
                            )

                            precision, recall, f1, specificity, fpr, fnr, error_rate, roc_auc = compute_metrics(
                                y_true,
                                y_pred,
                                y_proba
                            )

                            # =====================
                            # STORE RESULTS
                            # =====================

                            results.append({

                                "Model": model_name,

                                "Augmentation": aug_name,

                                "Freeze": freeze_name,

                                "Optimizer": opt_name,

                                "LR": lr,

                                "Batch Size": bs,

                                "Epochs": epochs,

                                "Accuracy": acc,

                                "Precision": precision,

                                "Recall": recall,

                                "F1": f1,

                                "Specificity": specificity,

                                "FPR": fpr,

                                "FNR": fnr,

                                "Error Rate": error_rate,

                                "ROC-AUC": roc_auc
                            })

                            # =====================
                            # PLOTS
                            # =====================

                            plot_all(
                                y_true,
                                y_pred,
                                y_proba,
                                history,
                                class_names,
                                title=file_name
                            )

# ============================================================
# 13. TRAIN CNN
# ============================================================

train_model(
    "CNN",
    build_cnn
)

# ============================================================
# 14. TRAIN TRANSFER MODELS
# ============================================================

for name, (model_fn, preprocess) in transfer_models.items():

    train_model(
        name,
        lambda num_classes,
               opt,
               lr,
               freeze:
               build_transfer_model(
                   model_fn,
                   num_classes,
                   opt,
                   lr,
                   freeze
               ),
        preprocess,
        transfer=True
    )

# ============================================================
# 15. SAVE RESULTS
# ============================================================

df = pd.DataFrame(results)

csv_path = os.path.join(
    RESULT_DIR,
    "results.csv"
)

excel_path = os.path.join(
    RESULT_DIR,
    "results.xlsx"
)

df.to_csv(csv_path, index=False)

df.to_excel(excel_path, index=False)

print("\n✅ Results Saved")
print(df.head())

Using Colab cache for faster access to the 'fruits-dataset-images' dataset.
✅ Dataset Path: /kaggle/input/fruits-dataset-images


Copying files: 360 files [00:00, 493.37 files/s]

✅ Dataset Split Completed
⏩ Already Trained -> CNN_NoAug_FineTune_Adam_lr0.01_bs24_ep5.keras
⏩ Already Trained -> CNN_NoAug_FineTune_RMSprop_lr0.0001_bs24_ep15.keras
⏩ Already Trained -> CNN_Augmented_FineTune_Adam_lr0.0001_bs24_ep10.keras
⏩ Already Trained -> CNN_Augmented_FineTune_RMSprop_lr0.0001_bs24_ep10.keras
⏩ Already Trained -> ResNet50_NoAug_Frozen_Adam_bs24.keras
⏩ Already Trained -> ResNet50_NoAug_Frozen_RMSprop_bs24.keras
⏩ Already Trained -> ResNet50_NoAug_FineTune_Adam_lr0.0001_bs24_ep10.keras
⏩ Already Trained -> ResNet50_NoAug_FineTune_RMSprop_lr0.0001_bs24_ep10.keras
⏩ Already Trained -> ResNet50_Augmented_Frozen_Adam_lr0.0001_bs24_ep10.keras
⏩ Already Trained -> ResNet50_Augmented_Frozen_RMSprop_lr0.0001_bs24_ep10.keras
⏩ Already Trained -> ResNet50_Augmented_FineTune_Adam_lr0.0001_bs24_ep15.keras
⏩ Already Trained -> ResNet50_Augmented_FineTune_RMSprop_lr0.0001_bs24_ep15.keras
Found 287 images belonging to 9 classes.
Found 72 images belonging to 9 classes.


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 52s 3s/step - accuracy: 0.1010 - loss: 2.7269 - val_accuracy: 0.0972 - val_loss: 2.1795
Epoch 2/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 12s 957ms/step - accuracy: 0.1951 - loss: 2.2304 - val_accuracy: 0.3333 - val_loss: 1.8773
Epoch 3/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 12s 1000ms/step - accuracy: 0.2787 - loss: 1.9103 - val_accuracy: 0.5278 - val_loss: 1.6639
Epoch 4/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 12s 971ms/step - accuracy: 0.4111 - loss: 1.7177 - val_accuracy: 0.5833 - val_loss: 1.4984
Epoch 5/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 12s 1s/step - accuracy: 0.5122 - loss: 1.4633 - val_accuracy: 0.6667 - val_loss: 1.3566
Epoch 6/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 11s 886ms/step - accuracy: 0.5958 - loss: 1.3498 - val_accuracy: 0.7222 - val_loss: 1.2299
Epoch 7/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 11s 957ms/step - accuracy: 0.6446 - loss: 1.2183 - val_accuracy: 0.7222 - val_loss: 1.1161
Epoch 8/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 12s 1s/step - a

3/3 ━━━━━━━━━━━━━━━━━━━━ 6s 173ms/step
Found 287 images belonging to 9 classes.
Found 72 images belonging to 9 classes.
Epoch 1/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 33s 2s/step - accuracy: 0.1638 - loss: 2.3468 - val_accuracy: 0.2917 - val_loss: 1.9524
Epoch 2/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 12s 1s/step - accuracy: 0.3066 - loss: 1.9387 - val_accuracy: 0.4444 - val_loss: 1.7086
Epoch 3/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 12s 946ms/step - accuracy: 0.3519 - loss: 1.7723 - val_accuracy: 0.5833 - val_loss: 1.5162
Epoch 4/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 11s 999ms/step - accuracy: 0.5540 - loss: 1.4567 - val_accuracy: 0.6528 - val_loss: 1.3470
Epoch 5/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 11s 836ms/step - accuracy: 0.6376 - loss: 1.2880 - val_accuracy: 0.6806 - val_loss: 1.2142
Epoch 6/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 12s 887ms/step - accuracy: 0.6690 - loss: 1.1455 - val_accuracy: 0.7500 - val_loss: 1.0996
Epoch 7/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 12s 998ms/step - accuracy: 0.7178 - loss: 1.0443 - val_accuracy: 0.7917 - 

In [ ]:
# ============================================================
# RECOVER RESULTS FROM SAVED MODELS
# ============================================================

import os
import pickle
import numpy as np
import pandas as pd

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score
)

from sklearn.preprocessing import label_binarize
import kagglehub

path = kagglehub.dataset_download(
    "shreyapmaher/fruits-dataset-images"
)

print("✅ Dataset Path:", path)

# ============================================================
# 4. ORIGINAL DATASET PATH
# ============================================================

dataset_path = "/kaggle/input/fruits-dataset-images/images"

# ============================================================
# 5. TRAIN-TEST SPLIT
# ============================================================

output_path = "/kaggle/working/fruits_split"

if not os.path.exists(output_path):

    splitfolders.ratio(
        dataset_path,
        output=output_path,
        seed=42,
        ratio=(0.8, 0.2)
    )

print("✅ Dataset Split Completed")


# ============================================================
# PATHS
# ============================================================

MODEL_DIR = "/content/drive/MyDrive/Fruits/models"

HISTORY_DIR = "/content/drive/MyDrive/Fruits/histories"

RESULT_DIR = "/content/drive/MyDrive/Fruits/results"

test_path = "/kaggle/working/fruits_split/val"

IMG_SIZE = (224,224)

# ============================================================
# TEST GENERATOR
# ============================================================

test_datagen = ImageDataGenerator(rescale=1./255)

test_gen = test_datagen.flow_from_directory(
    test_path,
    target_size=IMG_SIZE,
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

# ============================================================
# METRICS FUNCTION
# ============================================================

def compute_metrics(y_true, y_pred, y_proba):

    precision = precision_score(
        y_true,
        y_pred,
        average='macro'
    )

    recall = recall_score(
        y_true,
        y_pred,
        average='macro'
    )

    f1 = f1_score(
        y_true,
        y_pred,
        average='macro'
    )

    cm = confusion_matrix(y_true, y_pred)

    TP = np.diag(cm)

    FP = cm.sum(axis=0) - TP

    FN = cm.sum(axis=1) - TP

    TN = cm.sum() - (FP + FN + TP)

    specificity = np.mean(TN / (TN + FP))

    fpr = np.mean(FP / (FP + TN))

    fnr = np.mean(FN / (FN + TP))

    error_rate = np.mean(FP + FN) / np.sum(cm)

    roc_auc = roc_auc_score(
        label_binarize(
            y_true,
            classes=np.unique(y_true)
        ),
        y_proba,
        multi_class='ovr'
    )

    return (
        precision,
        recall,
        f1,
        specificity,
        fpr,
        fnr,
        error_rate,
        roc_auc
    )

# ============================================================
# RESULTS LIST
# ============================================================

results = []

# ============================================================
# LOOP THROUGH MODELS
# ============================================================

model_files = [
    f for f in os.listdir(MODEL_DIR)
    if f.endswith(".keras")
]

print(f"Total Models Found: {len(model_files)}")

for file_name in model_files:

    print(f"\nProcessing: {file_name}")

    model_path = os.path.join(MODEL_DIR, file_name)

    # ========================================================
    # LOAD MODEL
    # ========================================================

    model = load_model(model_path)

   # ========================================================
    # PREDICTIONS
    # ========================================================

    test_gen.reset()

    y_proba = model.predict(
        test_gen,
        verbose=0
    )
    y_pred = np.argmax(y_proba, axis=1)

    y_true = test_gen.classes

    # ========================================================
    # METRICS
    # ========================================================

    acc = accuracy_score(y_true, y_pred)

    (
        precision,
        recall,
        f1,
        specificity,
        fpr,
        fnr,
        error_rate,
        roc_auc
    ) = compute_metrics(
        y_true,
        y_pred,
        y_proba
    )

    # ========================================================
    # PARSE FILENAME
    # ========================================================

    name = file_name.replace(".keras", "")

    parts = name.split("_")
    # ========================================================
    # SKIP INVALID / GENERIC MODELS
    # ========================================================

    if len(parts) < 4:

        print(f"⚠️ Skipping model: {file_name}")

        continue

    model_name = parts[0]

    augmentation = parts[1]

    freeze = parts[2]

    optimizer = parts[3]

    lr = None
    bs = None
    epochs = None

    for p in parts:

        if p.startswith("lr"):
            lr = p.replace("lr", "")

        elif p.startswith("bs"):
            bs = p.replace("bs", "")

        elif p.startswith("ep"):
            epochs = p.replace("ep", "")

    # ========================================================
    # LOAD HISTORY
    # ========================================================

    history_file = file_name.replace(
        ".keras",
        ".pkl"
    )

    history_path = os.path.join(
        HISTORY_DIR,
        history_file
    )

    train_acc = None
    val_acc = None

    if os.path.exists(history_path):

        with open(history_path, "rb") as f:

            history = pickle.load(f)

        train_acc = history['accuracy'][-1]

        val_acc = history['val_accuracy'][-1]

    # ========================================================
    # STORE RESULTS
    # ========================================================

    results.append({

        "Model": model_name,

        "Augmentation": augmentation,

        "Freeze": freeze,

        "Optimizer": optimizer,

        "LR": lr,

        "Batch Size": bs,

        "Epochs": epochs,

        "Train Accuracy": train_acc,

        "Validation Accuracy": val_acc,

        "Accuracy": acc,

        "Precision": precision,

        "Recall": recall,

        "F1": f1,

        "Specificity": specificity,

        "FPR": fpr,

        "FNR": fnr,

        "Error Rate": error_rate,

        "ROC-AUC": roc_auc
    })

# ============================================================
# DATAFRAME
# ============================================================

df = pd.DataFrame(results)

# ============================================================
# SAVE RESULTS
# ============================================================

csv_path = os.path.join(
    RESULT_DIR,
    "recovered_results.csv"
)

excel_path = os.path.join(
    RESULT_DIR,
    "recovered_results.xlsx"
)

df.to_csv(csv_path, index=False)

df.to_excel(excel_path, index=False)

print("\n✅ Results Recovered Successfully")

print(df.head())

Using Colab cache for faster access to the 'fruits-dataset-images' dataset.
✅ Dataset Path: /kaggle/input/fruits-dataset-images
✅ Dataset Split Completed
Found 72 images belonging to 9 classes.
Total Models Found: 52

Processing: ResNet50_NoAug_Frozen_Adam_bs24.keras

Processing: best_model.keras
⚠️ Skipping model: best_model.keras

Processing: ResNet50_NoAug_Frozen_Adam_bs32.keras


Copying files: 120 files [01:56,  1.03 files/s] 



Processing: ResNet50_NoAug_Frozen_SGD_bs24.keras

Processing: ResNet50_NoAug_Frozen_SGD_bs32.keras

Processing: ResNet50_NoAug_Frozen_RMSprop_bs24.keras

Processing: ResNet50_NoAug_Frozen_RMSprop_bs32.keras

Processing: CNN_NoAug_FineTune_Adam_lr0.01_bs24_ep5.keras

Processing: CNN_NoAug_FineTune_Adam_lr0.01_bs24_ep10.keras

Processing: CNN_NoAug_FineTune_Adam_lr0.01_bs24_ep15.keras

Processing: CNN_NoAug_FineTune_Adam_lr0.001_bs24_ep10.keras

Processing: CNN_NoAug_FineTune_Adam_lr0.001_bs24_ep15.keras

Processing: CNN_NoAug_FineTune_Adam_lr0.001_bs32_ep10.keras

Processing: CNN_NoAug_FineTune_Adam_lr0.001_bs32_ep15.keras

Processing: CNN_NoAug_FineTune_Adam_lr0.0001_bs24_ep10.keras

Processing: CNN_NoAug_FineTune_Adam_lr0.0001_bs24_ep15.keras

Processing: CNN_NoAug_FineTune_Adam_lr0.0001_bs32_ep10.keras

Processing: CNN_NoAug_FineTune_Adam_lr0.0001_bs32_ep15.keras

Processing: CNN_NoAug_FineTune_SGD_lr0.0001_bs24_ep15.keras

Processing: CNN_NoAug_FineTune_SGD_lr0.0001_bs32_ep15.keras